In [2]:
!ls

sample_data


In [3]:
!git clone https://github.com/karpathy/nanoGPT

Cloning into 'nanoGPT'...
remote: Enumerating objects: 689, done.
remote: Total 689 (delta 0), reused 0 (delta 0), pack-reused 689 (from 1)
Receiving objects: 100% (689/689), 975.25 KiB | 3.74 MiB/s, done.
Resolving deltas: 100% (382/382), done.


In [4]:
%cd nanoGPT

/content/nanoGPT


In [5]:
!python data/shakespeare_char/prepare.py

length of dataset in characters: 1,115,394
all the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size: 65
train has 1,003,854 tokens
val has 111,540 tokens


In [6]:
!git clone https://github.com/tianocore/edk2

Cloning into 'edk2'...
remote: Enumerating objects: 419139, done.
remote: Counting objects: 100% (292/292), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 419139 (delta 191), reused 175 (delta 175), pack-reused 418847 (from 3)
Receiving objects: 100% (419139/419139), 330.93 MiB | 26.17 MiB/s, done.
Resolving deltas: 100% (305575/305575), done.
Updating files: 100% (9829/9829), done.


In [8]:
!pwd

/content/nanoGPT


In [14]:
%%writefile build_data.py
#!/usr/bin/env python3
import os
import sys

def list_tree(startpath: str):
    """
    Print all directories and files under startpath in a tree-like structure.
    """
    for root, dirs, files in os.walk(startpath):
        # Compute the depth to create the tree indent
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}📁 {os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f"{subindent}📄 {f}")

def collect_by_extension(startpath: str) -> dict:
    """
    Walk the directory tree and return a dict mapping file-extensions to lists of file-paths.
    Files without an extension are grouped under the key ''.
    """
    ext_map = {}
    for dirpath, _, filenames in os.walk(startpath):
        for fname in filenames:
            _, ext = os.path.splitext(fname)
            ext = ext.lower()  # normalize
            fullpath = os.path.join(dirpath, fname)
            ext_map.setdefault(ext, []).append(fullpath)
    return ext_map

def concatenate_files(file_list: list, output_path: str):
    """
    Given a list of file paths, read each in turn and append its contents
    into the output_path file, with a header between files.
    """
    with open(output_path, 'w', encoding='utf-8', errors='ignore') as out_f:
        for idx, fpath in enumerate(file_list, start=1):
            try:
                with open(fpath, 'r', encoding='utf-8', errors='ignore') as in_f:
                    out_f.write(f"/* ===== File {idx}: {fpath} ===== */\n")
                    out_f.write(in_f.read())
                    out_f.write("\n\n")
                print(f"[+] Appended: {fpath}")
            except Exception as e:
                print(f"[!] Skipped {fpath}: {e}")
"""
# Print tree and collect all .c files into all_sources.txt
python3 repo_tools.py ~/dev/linux .c all_sources.txt
"""
def main():
    """
    if len(sys.argv) < 2:
        print(f"Usage: {sys.argv[0]} <path-to-repo> [extension] [output-file]")
        print("  <path-to-repo> : root directory to scan")
        print("  [extension]    : file extension to collect (e.g. .c). Default: .c")
        print("  [output-file]  : where to write combined contents. Default: combined.txt")
        sys.exit(1)

    repo_path = sys.argv[1]
    ext = sys.argv[2] if len(sys.argv) > 2 else '.c'
    out_file = sys.argv[3] if len(sys.argv) > 3 else 'combined.txt'
    """

    repo_path = "/content/nanoGPT/edk2"
    ext = '.c'
    out_file = 'combined.txt'

    if not os.path.isdir(repo_path):
        print(f"Error: {repo_path} is not a directory.")
        sys.exit(1)

    # 1. Print the directory tree
    print("=== Repository Tree ===")
    list_tree(repo_path)

    # 2. Build extension map
    print("\n=== Gathering files ===")
    ext_map = collect_by_extension(repo_path)
    file_list = ext_map.get(ext.lower(), [])

    if not file_list:
        print(f"No files with extension '{ext}' found.")
        sys.exit(0)

    print(f"Found {len(file_list)} '{ext}' files. Writing to '{out_file}'...")

    # 3. Concatenate them
    concatenate_files(file_list, out_file)
    print("Done.")

if __name__ == '__main__':
    main()

Overwriting build_data.py


In [15]:
!python build_data.py

Strumieniowane dane wyjściowe obcięte do 5000 ostatnich wierszy.
            📁 CmockaLib/
                📄 CmockaLib.inf
                📄 CmockaLib.uni
                📁 cmocka/
            📁 UnitTestUefiBootServicesTableLib/
                📄 UnitTestUefiBootServicesTableLib.inf
                📄 UnitTestUefiBootServicesTableLibMemory.c
                📄 UnitTestUefiBootServicesTableLibProtocol.h
                📄 UnitTestUefiBootServicesTableLib.h
                📄 UnitTestUefiBootServicesTableLibEventTimer.c
                📄 UnitTestUefiBootServicesTableLibTpl.c
                📄 UnitTestUefiBootServicesTableLib.c
                📄 UnitTestUefiBootServicesTableLibMisc.c
                📄 UnitTestUefiBootServicesTableLibProtocol.c
                📄 UnitTestUefiBootServicesTableLib.uni
                📄 UnitTestUefiBootServicesTableLibImage.c
            📁 UnitTestLib/
                📄 Log.c
                📄 AssertCmocka.c
                📄 UnitTestLibCmocka.uni
                📄

In [21]:
!ls /content/nanoGPT/

assets	       configurator.py	part_0.txt	    train.py
bench.py       data		README.md	    transformer_sizing.ipynb
build_data.py  edk2		run_combined.py
combined.txt   LICENSE		sample.py
config	       model.py		scaling_laws.ipynb


In [18]:
!ls -ls /content/nanoGPT/

total 50520
    4 drwxr-xr-x  2 root root     4096 Mar 17 14:28 assets
    8 -rw-r--r--  1 root root     4815 Mar 17 14:28 bench.py
    4 -rw-r--r--  1 root root     3240 Mar 17 14:32 build_data.py
50152 -rw-r--r--  1 root root 51352378 Mar 17 14:33 combined.txt
    4 drwxr-xr-x  2 root root     4096 Mar 17 14:28 config
    4 -rw-r--r--  1 root root     1758 Mar 17 14:28 configurator.py
    4 drwxr-xr-x  5 root root     4096 Mar 17 14:28 data
    4 drwxr-xr-x 38 root root     4096 Mar 17 14:30 edk2
    4 -rw-r--r--  1 root root     1072 Mar 17 14:28 LICENSE
   16 -rw-r--r--  1 root root    16345 Mar 17 14:28 model.py
   16 -rw-r--r--  1 root root    13850 Mar 17 14:28 README.md
    4 -rw-r--r--  1 root root     3942 Mar 17 14:28 sample.py
  264 -rw-r--r--  1 root root   268519 Mar 17 14:28 scaling_laws.ipynb
   16 -rw-r--r--  1 root root    14857 Mar 17 14:28 train.py
   16 -rw-r--r--  1 root root    14579 Mar 17 14:28 transformer_sizing.ipynb


In [19]:
%%writefile run_combined.py
# split_large_file.py

def split_file(input_file='combined.txt', max_chunk_size=50 * 1024 * 1024):
    """
    Split a large file into multiple smaller files, each approximately max_chunk_size bytes.
    Default chunk size is 50 MB.
    """
    i = 0
    size = 0

    try:
        with open(input_file, 'r', encoding='utf-8', errors='ignore') as infile:
            out_file = open(f'part_{i}.txt', 'w', encoding='utf-8')
            for line in infile:
                line_size = len(line.encode('utf-8'))
                if size + line_size > max_chunk_size:
                    out_file.close()
                    i += 1
                    out_file = open(f'part_{i}.txt', 'w', encoding='utf-8')
                    size = 0
                out_file.write(line)
                size += line_size
            out_file.close()
        print(f"✅ Done. Created {i + 1} part files.")
    except FileNotFoundError:
        print(f"❌ File not found: {input_file}")
    except Exception as e:
        print(f"❌ Error occurred: {e}")

if __name__ == "__main__":
    split_file()

Writing run_combined.py


In [20]:
!python run_combined.py

✅ Done. Created 1 part files.


In [22]:
!mv combined.txt input.txt

In [23]:
!ls

assets		 data	    part_0.txt		train.py
bench.py	 edk2	    README.md		transformer_sizing.ipynb
build_data.py	 input.txt  run_combined.py
config		 LICENSE    sample.py
configurator.py  model.py   scaling_laws.ipynb


In [27]:
!ls -la /content/nanoGPT/data/shakespeare_char/

total 52356
drwxr-xr-x 2 root root     4096 Mar 17 14:29 .
drwxr-xr-x 5 root root     4096 Mar 17 14:28 ..
-rw-r--r-- 1 root root 51352378 Mar 17 14:39 input.txt
-rw-r--r-- 1 root root      703 Mar 17 14:29 meta.pkl
-rw-r--r-- 1 root root     2344 Mar 17 14:28 prepare.py
-rw-r--r-- 1 root root      209 Mar 17 14:28 readme.md
-rw-r--r-- 1 root root  2007708 Mar 17 14:29 train.bin
-rw-r--r-- 1 root root   223080 Mar 17 14:29 val.bin


In [26]:
!cp input.txt /content/nanoGPT/data/shakespeare_char/

In [36]:
!python train.py config/train_shakespeare_char.py

Overriding config with config/train_shakespeare_char.py:
# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-shakespeare-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = False

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 5000
lr_decay_iters = 5000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of 

In [30]:
!cat /content/nanoGPT/config/train_shakespeare_char.py

# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-shakespeare-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = False

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 5000
lr_decay_iters = 5000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of tokens per iter is small

warmup_iters = 100 # not super 

In [34]:
%%writefile /content/nanoGPT/config/train_shakespeare_char.py
# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-shakespeare-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = False

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 5000
lr_decay_iters = 5000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of tokens per iter is small

warmup_iters = 100 # not super necessary potentially

# on macbook also add
# device = 'cpu'  # run on cpu only
# compile = False # do not torch compile the model


Overwriting /content/nanoGPT/config/train_shakespeare_char.py


In [35]:
!python data/shakespeare_char/prepare.py

length of dataset in characters: 51,352,353
all the unique characters: 	
 !"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_`abcdefghijklmnopqrstuvwxyz{|}~ăΣ动序拟盘程虚键驱
vocab size: 107
train has 46,217,117 tokens
val has 5,135,236 tokens
